In [2]:
#/root/jupyter_notebooks/mvm-accelerator/test.ipynb

import pynq
from pynq import Overlay, allocate, MMIO, PL
import time, re, threading
import numpy as np

### Config

In [16]:
DEBUG   = 3 # (0, 1, 2, 3)
PROFILE = 1 # (0, 1)

In [4]:
PROJ_DIR  = "/home/ubuntu/mvm-accelerator/hw/"
BIT       = PROJ_DIR + "design_1.bit"
MTRX_PATH = "/home/ubuntu/mvm-accelerator/trmult_reduced.bin"

The following values are fixed in hardware:

In [5]:
CDMA_BASE = 0xA0000000 
BRAM_BASE = 0x80000000 

AXIS_CLK_HZ = 200e6

AVAILABLE_CHANNELS = 2
ACTIVE_CHANNELS = AVAILABLE_CHANNELS

NUM_ROWS = 1024
ROWS_PER_CHANNEL = NUM_ROWS // ACTIVE_CHANNELS

WORD_WIDTH = 64
ELEMENT_WIDTH = 64
ELEMENTS_PER_WORD = WORD_WIDTH // ELEMENT_WIDTH
BYTES_PER_WORD = WORD_WIDTH // 8

ELEMENTS_PER_ROW = 16384
WORDS_PER_ROW = ELEMENTS_PER_ROW // ELEMENTS_PER_WORD
BYTES_PER_ROW = WORDS_PER_ROW * BYTES_PER_WORD

NUM_PARTITIONS = AVAILABLE_CHANNELS

ELEMENTS_PER_PARTITION = ELEMENTS_PER_ROW // NUM_PARTITIONS
WORDS_PER_PARTITION = WORDS_PER_ROW // NUM_PARTITIONS
BYTES_PER_PARTITION = WORDS_PER_PARTITION * BYTES_PER_WORD

In [6]:
assert WORD_WIDTH % ELEMENT_WIDTH == 0
assert ELEMENTS_PER_ROW % ACTIVE_CHANNELS == 0
assert NUM_ROWS % ACTIVE_CHANNELS == 0

### Helpers

In [7]:
#https://github.com/iamNCJ/PYNQ-CDMA-Driver/blob/main/pynq_cdma/cdma.py

from pynq import DefaultIP
from pynq.buffer import PynqBuffer

class CDMA(DefaultIP):
    def __init__(self, description):
        super().__init__(description=description)

    bindto = ['xilinx.com:ip:axi_cdma:4.1']

    @property
    def idle(self):
        """
        `transfer` can only be called when the DMA is idle
        :return: True if the DMA engine is idle
        """
        return self.register_map.CDMASR.Idle

    def _do_transfer(self, source: int, dest: int, bytes_count: int):
        """
        Execute DMA transfer
        :param source: source address
        :param dest: destination address
        :param bytes_count: bytes to transfer
        :return: None
        """
        if not self.idle:
            raise Exception('DMA transfer can only start when engine is idle')

        assert source > 0
        if source > 0xFFFF_FFFF:
            self.register_map.SA_MSB[:] = source >> 8
        self.register_map.SA[:] = source & 0xFFFF_FFFF

        assert dest > 0
        if dest > 0xFFFF_FFFF:
            self.register_map.DA_MSB[:] = dest >> 8
        self.register_map.DA[:] = dest & 0xFFFF_FFFF

        assert bytes_count > 0
        self.register_map.BTT = bytes_count

        # Spin
        while not self.idle:
            pass

    def transfer(self, source, dest, bytes_count=None):
        """
        Transfer data through DMA
        :param source:
        :param dest:
        :param bytes_count:
        :return:
        """
        # Set source
        bytes_source = None
        if isinstance(source, PynqBuffer):
            source_addr = source.device_address
            bytes_source = source.nbytes
        elif isinstance(source, int):
            source_addr = source
        else:
            raise TypeError(f'Type {type(source)} not supported as source')

        bytes_dest = None
        if isinstance(dest, PynqBuffer):
            dest_addr = dest.device_address
            bytes_dest = dest.nbytes
        elif isinstance(dest, int):
            dest_addr = dest
        else:
            raise TypeError(f'Type {type(source)} not supported as dest')

        if bytes_count is None:
            if bytes_source is not None and bytes_dest is not None:
                bytes_count = min(bytes_source, bytes_dest)
            elif bytes_source is not None:
                bytes_count = bytes_source
            elif bytes_dest is not None:
                bytes_count = bytes_dest
            else:
                raise RuntimeError(f'Bytes to transfer is not set and cannot be inferred')

        self._do_transfer(source_addr, dest_addr, bytes_count)

In [19]:
from typing import Tuple, Dict, Any

try:
    from pynq import Overlay, MMIO
except ImportError:
    Overlay = None
    MMIO = None


class AxisDmaProfiler:
    """
    Address map:
    
        +0x00: STATUS [0]=have_result, [1]=running
        
        +0x04: CYCLES    [31:0]   +0x08: CYCLES    [63:32]
        +0x0C: BEATS     [31:0]   +0x10: BEATS     [63:32]
        +0x14: BYTES     [31:0]   +0x18: BYTES     [63:32]
        +0x1C: PKTS      [31:0]
        
        +0x20: GAP_LAST  [31:0]   +0x24: GAP_LAST  [63:32]
        +0x28: GAP_TOTAL [31:0]   +0x2C: GAP_TOTAL [63:32]
        +0x30: GAP_MIN   [31:0]   +0x34: GAP_MIN   [63:32]
        +0x38: GAP_MAX   [31:0]   +0x3C: GAP_MAX   [63:32]
        +0x40: GAP_COUNT [31:0]
        
        +0x44: GAP_STATUS [0]=gap_valid, [1]=armed
        
        +0x48: BUSY_TOTAL[31:0]   +0x4C: BUSY_TOTAL[63:32]
    """
    
    OFF_STATUS = 0x00
    OFF_CYCLES = 0x04
    OFF_BEATS  = 0x0C
    OFF_BYTES  = 0x14
    OFF_PKTS   = 0x1C
    
    OFF_GAP_LAST   = 0x20
    OFF_GAP_TOTAL  = 0x28
    OFF_GAP_MIN    = 0x30
    OFF_GAP_MAX    = 0x38
    OFF_GAP_COUNT  = 0x40
    OFF_GAP_STATUS = 0x44
    OFF_BUSY_TOTAL = 0x48
    
    CH_WINDOW = 0x100 # 0x100 bytes per channel

    def __init__(self, overlay, channels=AVAILABLE_CHANNELS, ip_name_contains="axis_dma_profiler"):
        self.channels = channels
        self.overlay = overlay

        key = next((k for k in overlay.ip_dict if ip_name_contains in k), None)
        
        if key is None:
            raise ValueError(
                f"Could not find IP containing '{ip_name_contains}' in overlay.ip_dict. "
                f"Pass base=... explicitly or adjust ip_name_contains."
            )
        ip = overlay.ip_dict[key]
        base = int(ip["phys_addr"])
        size = int(ip["addr_range"])

        if base is None: raise ValueError("Profiler base address is unknown.")
        if size is None: raise ValueError("Profiler size address is unknown.")

        self.base = base
        self.size = size
        self.mmio = MMIO(self.base, self.size)
        
        if DEBUG: print(f"    {self.__repr__()}")

    def _rd32(self, off):
        return self.mmio.read(off)

    def _rd64_stable(self, off, attempts=4):
        for _ in range(attempts):
            hi1 = self._rd32(off + 4)
            lo  = self._rd32(off)
            hi2 = self._rd32(off + 4)
            if hi1 == hi2:
                return (hi1 << 32) | lo
        return -1

    def _ch_base(self, ch: int) -> int:
        if not (0 <= ch < self.channels):
            raise ValueError(f"Channel out of range: {ch} (channels={self.channels})")
        return ch * self.CH_WINDOW

    # ========== Public ==========

    def stats(self, ch) -> Dict[str, int]:
        """
        Reads (cycles, beats, bytes, pkts). Prefer calling when have_result=1 and running=0.
        """
        b = self._ch_base(ch)
        cycles = self._rd64_stable(b + self.OFF_CYCLES)
        beats  = self._rd64_stable(b + self.OFF_BEATS)
        bytes_ = self._rd64_stable(b + self.OFF_BYTES)
        pkts   = self._rd32(b + self.OFF_PKTS)
        
        return {
            "cycles": cycles, 
            "beats":  beats, 
            "bytes":  bytes_, 
            "pkts":   pkts
        }
    
    def gap_stats(self, ch) -> Dict[str, Any]:
        """
        Inter-packet gap metrics for channel ch.
        """
        b = self._ch_base(ch)
        gap_last   = self._rd64_stable(b + self.OFF_GAP_LAST)
        gap_total  = self._rd64_stable(b + self.OFF_GAP_TOTAL)
        gap_min    = self._rd64_stable(b + self.OFF_GAP_MIN)
        gap_max    = self._rd64_stable(b + self.OFF_GAP_MAX)
        gap_count  = self._rd32(b + self.OFF_GAP_COUNT)
        busy_total = self._rd64_stable(b + self.OFF_BUSY_TOTAL)

        return {
            "gap_last":   gap_last,
            "gap_total":  gap_total,
            "gap_min":    gap_min,
            "gap_max":    gap_max,
            "gap_count":  gap_count,
            "busy_total": busy_total
        }

    def throughput(self, ch, axis_clk_hz=AXIS_CLK_HZ) -> Tuple[float, float, Dict[str, int]]:
        """
        Returns (gbps, handshake_utilization, stats) for a completed frame on channel ch.
        - gbps  = (bytes * 8 / cycles) * axis_clk_hz / 1e9
        - handshake_utilization = beats / cycles   (fraction of cycles that handshook)
        """
        s = self.stats(ch)
        cycles = max(1, s["cycles"])
        gbps = (s["bytes"] * 8.0 * axis_clk_hz / cycles) / 1e9
        util = s["beats"] / cycles
        return gbps, util, s
    
    def gap_summary(self, ch, axis_clk_hz=AXIS_CLK_HZ):
        """
        Analogous to `throughput`: return (g, mean_gap_ns, util_long_run).
          - g: dict with gap aggregates (gap_last, gap_min/max, gap_total, gap_count, busy_total)
          - mean_gap_ns: average inter-packet gap in nanoseconds
          - util_long_run: busy_total / (busy_total + gap_total)
        """
        g = self.gap_stats(ch)
        gap_total  = g["gap_total"]
        gap_count  = g["gap_count"]
        busy_total = g["busy_total"]

        mean_gap_cycles = (gap_total / gap_count) if gap_count else 0.0
        mean_gap_ns     = (1e9 * mean_gap_cycles / axis_clk_hz) if axis_clk_hz else 0.0
        util_long_run   = (busy_total / (busy_total + gap_total)) if (busy_total + gap_total) else 0.0

        return g, int(mean_gap_cycles), int(mean_gap_ns), util_long_run

    def __repr__(self) -> str:
        return f"AxisDmaProfiler(base=0x{self.base:08x}, size=0x{self.size:x}, channels={self.channels})"


In [9]:
def allocate_buf(dim1, dim2=None):
    if dim2 is not None:
        return allocate(shape=(dim1, dim2), dtype=np.float64, cacheable=False)
    else:
        return allocate(shape=(dim1,), dtype=np.float64, cacheable=False)

In [10]:
def get_dmas(overlay):
    dmas = [
        getattr(overlay, name)
        for name in sorted(overlay.ip_dict, key=lambda n: int(re.search(r"\d+$", n).group()))
        if name.startswith("axi_dma")
    ]
    if DEBUG: print("    dmas", [d.description['fullpath'] for d in dmas])
    if len(dmas) != AVAILABLE_CHANNELS:
        raise RuntimeError(f"Check overlay. Found {len(dmas)} DMAs with AVAILABLE_CHANNELS={AVAILABLE_CHANNELS}.")
    return dmas

In [26]:
def write_vec(overlay, vector):
    # Write vector to BRAM    
    if DEBUG: print("Writing vector to BRAM.")
    cdma = CDMA(overlay.ip_dict['axi_cdma_0'])

    # Create dummy vector
    for i in range(NUM_PARTITIONS):
        vector[i, :] = np.arange(
            i * ELEMENTS_PER_PARTITION + 1,
            (i + 1) * ELEMENTS_PER_PARTITION + 1,
            dtype=np.float64) / 10000.0

    for i in range(NUM_PARTITIONS):
        dest = BRAM_BASE + i * BYTES_PER_PARTITION
        if DEBUG: print(f"    vec[{i}]: src_addr=0x{vector[i].physical_address:8x}, dest_addr=0x{dest:8x}")
        cdma.transfer(vector[i], dest)

In [27]:
def create_workers(dmas, matrix):
    if DEBUG: print("Creating worker threads.")
    workers = []
    
    for ch in range(ACTIVE_CHANNELS):
        w = threading.Thread(
            target=worker,
            args=(ch, dmas[ch], matrix[ch]),
            daemon=True
        )
        workers.append(w)
        if DEBUG: print(f"    Initialized worker {ch}")
            
    return workers

In [28]:
def worker(ch, dma, matrix_chunk):
    if DEBUG > 1: print(f"[WORKER {ch}] Starting up...")
    dma.sendchannel.transfer(matrix_chunk)
    dma.sendchannel.wait()
    if DEBUG > 1: print(f"[WORKER {ch}] Done.")

### Main

In [29]:
if __name__ == "__main__":
    if DEBUG: 
        print(f"\n================================ Parameters =================================")
        
        print(f"""
    CDMA_BASE = 0x{CDMA_BASE:08x}
    BRAM_BASE = 0x{BRAM_BASE:08x}

    NUM_ROWS           = {NUM_ROWS}
    AVAILABLE_CHANNELS = {AVAILABLE_CHANNELS} 
    ACTIVE_CHANNELS    = {ACTIVE_CHANNELS}
    ROWS_PER_CHANNEL   = {ROWS_PER_CHANNEL}

    WORD_WIDTH        = {WORD_WIDTH}
    ELEMENT_WIDTH     = {ELEMENT_WIDTH}
    ELEMENTS_PER_WORD = {ELEMENTS_PER_WORD}
    BYTES_PER_WORD    = {BYTES_PER_WORD}

    ELEMENTS_PER_ROW = {ELEMENTS_PER_ROW}
    WORDS_PER_ROW    = {WORDS_PER_ROW}
    BYTES_PER_ROW    = {BYTES_PER_ROW}

    NUM_PARTITIONS = {NUM_PARTITIONS}

    ELEMENTS_PER_PARTITION = {ELEMENTS_PER_PARTITION}
    WORDS_PER_PARTITION    = {WORDS_PER_PARTITION}
    BYTES_PER_PARTITION    = {BYTES_PER_PARTITION}""")

    PL.reset()
    matrix = vector = result = None 

    try:
        if DEBUG: print("\n================================ Initialize =================================\n")

        # Load overlay
        if DEBUG: print("Loading Overlay.")
        overlay = Overlay(BIT, download=True)
        if not overlay.is_loaded():
            raise RuntimeError("Overlay download failed.")
        
        # Get DMAs and Profiler
        dmas = get_dmas(overlay)
        if PROFILE: prof = AxisDmaProfiler(overlay)

        # Allocate buffers
        if DEBUG: print("Allocating buffers.")
        matrix = [allocate_buf(ROWS_PER_CHANNEL, ELEMENTS_PER_ROW) for _ in range(ACTIVE_CHANNELS)]
        vector =  allocate_buf(NUM_PARTITIONS, ELEMENTS_PER_PARTITION)
        result = [allocate_buf(ROWS_PER_CHANNEL) for _ in range(ACTIVE_CHANNELS)]
        if DEBUG:
            [print(f"    matrix[{i}]: src_addr=0x{m.physical_address:08x}") for i,m in enumerate(matrix)]
            print(f"    vector: src_addr=0x{vector.physical_address:08x}")
            [print(f"    result[{i}]: src_addr=0x{r.physical_address:08x}") for i,r in enumerate(result)]

        # Read matrix into buffer
        if DEBUG: print("Reading matrix file.")
        with open(MTRX_PATH, "rb") as a_file:
            for i in range(ACTIVE_CHANNELS):
                for j in range(ROWS_PER_CHANNEL):
                    view = memoryview(matrix[i][j]).cast("B")
                    if a_file.readinto(view) != BYTES_PER_ROW:
                        raise RuntimeError(f"Failed to read full row {i}.")
                    if DEBUG > 2: 
                        print(f"    matrix[{i}][{j}]: [", end=" ")
                        for k in range(3): print(f"{matrix[i][j][k]},", end=" ")
                        print(f"... ]")
                        
        # Write vector to BRAM
        write_vec(overlay, vector)

        # Create worker threads
        workers = create_workers(dmas, matrix)

        # Execute
        if DEBUG: print("\n================================== Compute ==================================\n")
        
        t0_comp = time.perf_counter()
        
        for ch in range(ACTIVE_CHANNELS): dmas[ch].recvchannel.transfer(result[ch])

        for w in workers: w.start()
        for w in workers: w.join()

        #for ch in range(ACTIVE_CHANNELS): dmas[ch].recvchannel.wait()

        t1_comp = time.perf_counter()

        # Print results
        if DEBUG:
            for i in range(ACTIVE_CHANNELS):
                for j in range(ROWS_PER_CHANNEL):
                    row_idx = i * ROWS_PER_CHANNEL + j
                    if (row_idx > 5 and row_idx < NUM_ROWS-5):
                        if not DEBUG > 2: # Only print all rows if DEBUG == 3
                            if (row_idx == 6): print("...")
                            continue
                    actual = result[i][j]
                    expected = float(np.dot(matrix[i][j], vector.flatten()))
                    print(f"Row {row_idx}: Actual={actual:.32f} | Expected={expected:.32f}")
            
            print("\n================================ Performance ================================\n")

            if PROFILE:
                for i in range(AVAILABLE_CHANNELS):
                    gbps, util, s = prof.throughput(ch=i)
                    g, mean_gap_cycles, mean_gap_ns, util_long_run = prof.gap_summary(ch=i)
                    print(f"Channel {i}:")
                    if DEBUG > 1:
                        print(f"    {s}")
                        print(f"    {g}")
                    print(f"    Throughput: {gbps:.3f} Gbps | Handshake util: {util:.2%}")
                    print(f"    Avg gap length: {mean_gap_cycles} cycles, {mean_gap_ns} ns | Busy util: {util_long_run:.2%}")
        
        print(f"\nCompute time: {(t1_comp - t0_comp) * 1000:.6f}ms")

    finally:
        for buf in [matrix, result]:
            if buf is not None:
                for b in buf: b.freebuffer()
        if vector is not None: vector.freebuffer()


================================ Parameters =================================

    CDMA_BASE = 0xa0000000
    BRAM_BASE = 0x80000000

    NUM_ROWS           = 1024
    AVAILABLE_CHANNELS = 2 
    ACTIVE_CHANNELS    = 2
    ROWS_PER_CHANNEL   = 512

    WORD_WIDTH        = 64
    ELEMENT_WIDTH     = 64
    ELEMENTS_PER_WORD = 1
    BYTES_PER_WORD    = 8

    ELEMENTS_PER_ROW = 16384
    WORDS_PER_ROW    = 16384
    BYTES_PER_ROW    = 131072

    NUM_PARTITIONS = 2

    ELEMENTS_PER_PARTITION = 8192
    WORDS_PER_PARTITION    = 8192
    BYTES_PER_PARTITION    = 65536

================================ Initialize =================================

Loading Overlay.
    dmas ['axi_dma_0', 'axi_dma_1']
    AxisDmaProfiler(base=0xa0040000, size=0x1000, channels=2)
Allocating buffers.
    matrix[0]: src_addr=0x39300000
    matrix[1]: src_addr=0x3d300000
    vector: src_addr=0x375c0000
    result[0]: src_addr=0x05ffc000
    result[1]: src_addr=0x78b16000
Reading matrix file.
    matrix[0][0]: [

    matrix[0][295]: [ 1.0625979584108957e-21, 1.3282036832585695e-21, 1.5755910972403634e-21, ... ]
    matrix[0][296]: [ 9.798689317749873e-21, 9.34705960531723e-17, 5.821207969199523e-17, ... ]
    matrix[0][297]: [ 4.043363737969074e-21, 5.2612083680224596e-21, 6.493722153497778e-21, ... ]
    matrix[0][298]: [ 1.3560696161313723e-21, 1.7762967822244463e-21, 2.3566844162447474e-21, ... ]
    matrix[0][299]: [ 9.972838344249588e-20, 6.831675942349694e-20, 4.4036195865604106e-20, ... ]
    matrix[0][300]: [ 1.1231265173974234e-18, 2.0396886141047173e-20, 1.649463229507015e-20, ... ]
    matrix[0][301]: [ 2.533167395799042e-19, 1.6948975097797686e-19, 1.7011103672277356e-19, ... ]
    matrix[0][302]: [ 2.314994248825522e-21, 2.503477947737881e-21, 2.998018869825611e-21, ... ]
    matrix[0][303]: [ 1.3462279277584079e-20, 1.67268404549247e-20, 1.9383674864466313e-20, ... ]
    matrix[0][304]: [ 6.990181226082396e-20, 8.666239219428369e-20, 9.391660604579918e-20, ... ]
    matrix[0][305]

    matrix[1][193]: [ 1.0894931632982888e-21, 9.244161321566612e-22, 7.5153592541056e-22, ... ]
    matrix[1][194]: [ 1.7746057706677222e-20, 2.3116463905679483e-20, 2.454914659624198e-20, ... ]
    matrix[1][195]: [ 5.144950929322677e-22, 5.785422917680107e-22, 5.505628807745021e-22, ... ]
    matrix[1][196]: [ 2.651223018320369e-19, 2.246970193351538e-19, 2.002506047630533e-19, ... ]
    matrix[1][197]: [ 2.6943315480151087e-18, 1.836661525459124e-18, 1.5152634508467203e-18, ... ]
    matrix[1][198]: [ 8.691212080664192e-21, 8.52297851441777e-21, 9.811730450978622e-21, ... ]
    matrix[1][199]: [ 1.9658534051286825e-15, 1.124222616863679e-17, 1.7281750503233276e-17, ... ]
    matrix[1][200]: [ 3.3696212065896813e-21, 3.5855692447816454e-21, 4.24870757044603e-21, ... ]
    matrix[1][201]: [ 1.506758086771582e-16, 8.247408293422961e-17, 6.701368442805196e-17, ... ]
    matrix[1][202]: [ 3.418003081161819e-17, 1.1758061035104331e-17, 6.005406992932826e-18, ... ]
    matrix[1][203]: [ 1.

[WORKER 1] Done.
[WORKER 0] Done.
Row 0: Actual=0.00000025988997789310375988682173 | Expected=0.00000025988997789310074233194714
Row 1: Actual=0.00288655128138357378042577749966 | Expected=0.00288655128138344844665463817535
Row 2: Actual=0.00012291017716705096855973888825 | Expected=0.00012291017716704720095718950112
Row 3: Actual=0.00003577723174499462696586046806 | Expected=0.00003577723174499375282785890162
Row 4: Actual=0.00010396862328385418020731956190 | Expected=0.00010396862328385100891596504180
Row 5: Actual=0.00180500806538218174640553215227 | Expected=0.00180500806538211951320083148431
Row 6: Actual=0.01332210137814396171129782686648 | Expected=0.01332210137814371364584076218307
Row 7: Actual=0.01214519443410199718569320026518 | Expected=0.01214519443410174738551265960496
Row 8: Actual=0.01674574629254114987997326124969 | Expected=0.01674574629254089314089881668224
Row 9: Actual=0.02004323601752783928775514254994 | Expected=0.02004323601752757560978679407526
Row 10: Actual=0

Row 139: Actual=0.00013479716967750945987893074207 | Expected=0.00013479716967750409307817693882
Row 140: Actual=0.00027737339400329377898934968272 | Expected=0.00027737339400328472590120942876
Row 141: Actual=0.00019474145571932391526014105576 | Expected=0.00019474145571932012055253735650
Row 142: Actual=0.04146644872654626362562524377608 | Expected=0.04146644872654554198065923742433
Row 143: Actual=0.05395031268735046847684344584195 | Expected=0.05395031268734992030422503717091
Row 144: Actual=0.00510340579128381113815926539701 | Expected=0.00510340579128370532002723081177
Row 145: Actual=0.04213834505853502471195426437589 | Expected=0.04213834505853453898938099086990
Row 146: Actual=0.01857306135689140019762177757912 | Expected=0.01857306135689113998910038105805
Row 147: Actual=0.00023043357620398089878263614416 | Expected=0.00023043357620398141377866807478
Row 148: Actual=0.00023945036565235598832387942103 | Expected=0.00023945036565235577148344492393
Row 149: Actual=0.000267271916

...

Row 419: Actual=0.00028848123942081302763587591365 | Expected=0.00028848123942079952931882846912
Row 420: Actual=0.00036873881026049901447547152422 | Expected=0.00036873881026048795561331217208
Row 421: Actual=0.00055888822745925402068339904460 | Expected=0.00055888822745923374610277356567
Row 422: Actual=0.03035755126730555267244859862785 | Expected=0.03035755126730517450273083568391
Row 423: Actual=0.00015677757944365379146403616240 | Expected=0.00015677757944364977991599796603
Row 424: Actual=0.00016677456632713564491782287647 | Expected=0.00016677456632713190442032780147
Row 425: Actual=0.00022331707486250842119822335174 | Expected=0.00022331707486250538543214039233
Row 426: Actual=0.00039958109547892834735594025375 | Expected=0.00039958109547892352265627269325
Row 427: Actual=0.00039483255962063814985263388024 | Expected=0.00039483255962063234937101108279
Row 428: Actual=0.00020320632267401884420154889455 | Expected=0.00020320632267401713658312722988
Row 429: Actual=0.000702794356

Row 559: Actual=0.00157437937596863132723445932726 | Expected=0.00157437937596862330413838293452
Row 560: Actual=0.00896692493005193415334108664183 | Expected=0.00896692493005187170329595147678
Row 561: Actual=0.06599209323509203084512364512193 | Expected=0.06599209323509164226706502631714
Row 562: Actual=0.06874916377019946178705822603661 | Expected=0.06874916377019939239811918696432
Row 563: Actual=0.00207308822508654169367026121051 | Expected=0.00207308822508654212735113020472
Row 564: Actual=0.00000000004058754507492513151442 | Expected=0.00000000004058754507492506042858
Row 565: Actual=0.00001177621106650929562703437881 | Expected=0.00001177621106650930748549564037
Row 566: Actual=0.00004501239046262351945881102777 | Expected=0.00004501239046262225229752193534
Row 567: Actual=0.00008977188375964309475715796616 | Expected=0.00008977188375963821584738178139
Row 568: Actual=0.00004604496800855965035976016808 | Expected=0.00004604496800855799017518354965
Row 569: Actual=0.000102393963

Row 699: Actual=0.00020104546879545144339070106287 | Expected=0.00020104546879544323055924448518
Row 700: Actual=0.00011419307351028752032899571400 | Expected=0.00011419307351028414574973385287
Row 701: Actual=0.00017968591354265356428536459799 | Expected=0.00017968591354264703196727537282
Row 702: Actual=0.00021155570536069090653921720424 | Expected=0.00021155570536068572947384358596
Row 703: Actual=0.00010382203891079169928452624605 | Expected=0.00010382203891078885325382347160
Row 704: Actual=0.00013073794171922207174758867598 | Expected=0.00013073794171921797888438754320
Row 705: Actual=0.00012481654406259980096688400053 | Expected=0.00012481654406259698204123553822
Row 706: Actual=0.00041225351639670579015350959651 | Expected=0.00041225351639670112808416790884
Row 707: Actual=0.00265400997481535097857752170114 | Expected=0.00265400997481529156429846949550
Row 708: Actual=0.01236851160359780932296303035400 | Expected=0.01236851160359758033946420141547
Row 709: Actual=0.062972420948

Row 839: Actual=0.00402850577131551486342875278979 | Expected=0.00402850577131540037167933832052
Row 840: Actual=0.00042004339403431200468050366048 | Expected=0.00042004339403431178784006916338
Row 841: Actual=0.00116898769475302140991024302252 | Expected=0.00116898769475302119306980852542
Row 842: Actual=0.00112303181829394478630224085691 | Expected=0.00112303181829394153369572340040
Row 843: Actual=0.00248893901976423052510578592944 | Expected=0.00248893901976421534627537113238
Row 844: Actual=0.00354567814569961732573677437586 | Expected=0.00354567814569961775941764337006
Row 845: Actual=0.00087520843630539300536663738228 | Expected=0.00087520843630539311378685463083
Row 846: Actual=0.00000000000395009290300848036588 | Expected=0.00000000000395009290300848521264
Row 847: Actual=0.00003828030568514316576544248560 | Expected=0.00003828030568514191893294412727
Row 848: Actual=0.00245717553048758335920709328093 | Expected=0.00245717553048741682575339950745
Row 849: Actual=0.007943577364

Row 979: Actual=0.00465303543466116884680383591899 | Expected=0.00465303543466098323139190640063
Row 980: Actual=0.00818275311923031233030112474580 | Expected=0.00818275311923001916203368466540
Row 981: Actual=0.00008718404899432058798892930529 | Expected=0.00008718404899431808077140543256
Row 982: Actual=0.00198586814591664552812066979470 | Expected=0.00198586814591655402145731201813
Row 983: Actual=0.00010456906541578954667002371304 | Expected=0.00010456906541578595525032735480
Row 984: Actual=0.00011485721020320515837684349769 | Expected=0.00011485721020320178379758163656
Row 985: Actual=0.00012480437184044572917172855053 | Expected=0.00012480437184044036237097474729
Row 986: Actual=0.00013440072633251048126248883285 | Expected=0.00013440072633250560235271264808
Row 987: Actual=0.00014366789667090033638316426412 | Expected=0.00014366789667089637904523469203
Row 988: Actual=0.00015259831767638735739583966122 | Expected=0.00015259831767638356268823596196
Row 989: Actual=0.000411922463